# 03 - Smoke Test (Small Data)

Quick validation: 100 examples, 1 epoch. ~5 minutes on T4.

**Purpose:** Verify the entire pipeline works before running the full training.

**Check:**
- [ ] Unsloth installs without error
- [ ] Model loads in 4-bit (~3.5GB VRAM)
- [ ] LoRA adapters apply correctly (~30M trainable params)
- [ ] Data loads and tokenizes
- [ ] Training loss decreases
- [ ] Model generates coherent output after training
- [ ] LoRA adapter saves correctly

In [ ]:
%%capture
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import torch

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9

print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA capability: {props.major}.{props.minor}")

assert props.major >= 7, f"GPU {gpu_name} (sm_{props.major}{props.minor}) not supported. Select T4 in Kaggle settings."
print("GPU check passed.")

In [ ]:
import os
os.chdir('/kaggle/working')

# Upload src/ as a Kaggle dataset and mount, or clone from GitHub
if not os.path.exists('fingpt-qlora'):
    !git clone https://github.com/W-Kaski/fingpt-qlora.git
os.chdir('fingpt-qlora')
print(f"Working dir: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

## 1. Load Model (Checkpoint 1)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

print(f"Model loaded: {MODEL_NAME}")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Checkpoint 1 PASS: Model loads in 4-bit.")

## 2. Apply LoRA (Checkpoint 2)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

assert trainable > 1_000_000, f"Too few trainable params: {trainable}"
assert trainable < 100_000_000, f"Too many trainable params: {trainable}"
print("Checkpoint 2 PASS: LoRA applied correctly.")

## 3. Prepare Small Dataset (Checkpoint 3)

In [ ]:
import json
from datasets import Dataset

# First: run data pipeline to generate splits (if not exists)
if not os.path.exists('data/splits/train.json'):
    print("Running data pipeline...")
    !python -m src.data.download
    !python -m src.data.preprocess
    !python -m src.data.format_chat
    !python -m src.data.merge_datasets
    !python -m src.data.splits

# Load only 100 training examples for smoke test
SMOKE_TEST_SIZE = 100

with open('data/splits/train.json') as f:
    full_train = json.load(f)
with open('data/splits/val.json') as f:
    full_val = json.load(f)

train_data = full_train[:SMOKE_TEST_SIZE]
val_data = full_val[:20]  # 20 val examples

print(f"Smoke test: {len(train_data)} train, {len(val_data)} val")

# Convert to HF Dataset with chat template
def to_dataset(data):
    texts = []
    for ex in data:
        messages = []
        for turn in ex['conversations']:
            role = turn['from']
            if role == 'gpt':
                role = 'assistant'
            messages.append({'role': role, 'content': turn['value']})
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        texts.append({'text': text})
    return Dataset.from_list(texts)

train_dataset = to_dataset(train_data)
val_dataset = to_dataset(val_data)

# Sanity check
sample = train_dataset[0]['text']
print(f"Sample length: {len(sample)} chars")
assert len(sample) > 50, "Sample too short - data formatting may be broken"
assert '<|im_start|>' in sample, "Missing ChatML tokens - chat template may be wrong"
print(f"Sample preview: {sample[:200]}...")
print("Checkpoint 3 PASS: Data loads and tokenizes correctly.")

## 4. Train 1 Epoch (Checkpoint 4)

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="outputs/smoke_test",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="adamw_8bit",
    max_seq_length=MAX_SEQ_LENGTH,
    fp16=True,
    bf16=False,
    logging_steps=5,
    save_strategy="no",  # Don't save checkpoints during smoke test
    eval_strategy="no",  # Skip eval to save time
    seed=42,
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print(f"Starting smoke test: {len(train_dataset)} examples, 1 epoch")
print(f"Steps: {len(train_dataset) // (2 * 4)}")
print(f"Estimated time: ~2 minutes")
print("="*50)

trainer.train()

# Check loss decreased
log_history = trainer.state.log_history
losses = [l['loss'] for l in log_history if 'loss' in l]
if len(losses) >= 2:
    first_loss = losses[0]
    last_loss = losses[-1]
    print(f"\nLoss: {first_loss:.4f} -> {last_loss:.4f}")
    assert last_loss < first_loss, f"Loss did not decrease! {first_loss} -> {last_loss}"
    print("Checkpoint 4 PASS: Training loss decreases.")
else:
    print("Warning: Not enough loss entries to verify decrease.")

## 5. Generate Output (Checkpoint 5)

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Analyze the sentiment: Tesla shares surged 12% after record Q3 deliveries",
    "What are the key risks of investing in emerging market bonds?",
]

for i, prompt in enumerate(test_prompts):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    
    print(f"\n{'='*50}")
    print(f"Prompt: {prompt}")
    print(f"{'='*50}")
    print(response)

print("\nCheckpoint 5 PASS: Model generates output after training.")

## 6. Save Adapter (Checkpoint 6)

In [ ]:
save_path = "outputs/smoke_test/lora_adapter"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# Verify files exist
saved_files = os.listdir(save_path)
print(f"Saved files: {saved_files}")

adapter_size = sum(os.path.getsize(os.path.join(save_path, f)) for f in saved_files)
print(f"Adapter size: {adapter_size / 1024 / 1024:.1f} MB")

assert adapter_size > 1_000_000, "Adapter seems too small"
assert adapter_size < 500_000_000, "Adapter seems too large"
print("Checkpoint 6 PASS: LoRA adapter saved correctly.")

## Summary

All 6 checkpoints passed:
1. Model loads in 4-bit
2. LoRA applies correctly (~30M params)
3. Data loads and tokenizes (ChatML format)
4. Training loss decreases
5. Model generates coherent output
6. Adapter saves correctly (~50MB)

**Safe to proceed with full training (03_training_qlora.ipynb).**

In [ ]:
# Final summary
print("="*50)
print("SMOKE TEST COMPLETE")
print("="*50)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Peak VRAM: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print(f"Trainable params: {trainable:,}")
print(f"Training examples: {len(train_data)}")
print(f"Final loss: {last_loss:.4f}")
print(f"Adapter size: {adapter_size / 1024 / 1024:.1f} MB")
print("\nReady for full training!")